# 🌸 RTAI-242P | Practical 7
## Package and Reproduce an AI Model with Docker & MLflow

**Objectives:**
- Train a model on Iris dataset
- Track experiment with MLflow
- Inspect `conda.yaml` and `requirements.txt` auto-generated by MLflow
- Understand Docker containerization concepts
- Test reproducibility via FastAPI

---

## Step 1: Install & Import Libraries

In [ ]:
# Install if not already installed
# !pip install mlflow scikit-learn fastapi uvicorn requests pandas

import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os, json

print('✅ All libraries imported successfully!')
print(f'   MLflow version : {mlflow.__version__}')

## Step 2: Load Dataset

In [ ]:
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name='species')

print('Dataset Info:')
print(f'  Shape   : {X.shape}')
print(f'  Classes : {list(iris.target_names)}')
X.head()

## Step 3: Configure MLflow

In [ ]:
mlflow.set_tracking_uri('mlruns')                  # Save runs locally
mlflow.set_experiment('iris-classification')        # Experiment name
print('✅ MLflow configured. Runs will be saved to: ./mlruns')

## Step 4: Train Model & Log with MLflow

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

with mlflow.start_run(run_name='notebook-run') as run:
    # Train
    model = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
    model.fit(X_train, y_train)

    # Evaluate
    y_pred   = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    # Log to MLflow
    mlflow.log_param('n_estimators', 100)
    mlflow.log_param('max_depth',    4)
    mlflow.log_metric('accuracy',    accuracy)

    # Save model
    mlflow.sklearn.log_model(model, 'iris-model', registered_model_name='IrisClassifier')

    run_id = run.info.run_id
    with open('run_id.txt', 'w') as f:
        f.write(run_id)

print(f'✅ Training Complete!')
print(f'   Accuracy : {accuracy:.4f}  ({accuracy*100:.2f}%)')
print(f'   Run ID   : {run_id}')

## Step 5: Inspect MLflow Auto-Generated Files

> **This is the key concept of the practical!**
> MLflow automatically creates `conda.yaml` and `requirements.txt` inside the model folder.
> These files make the model **reproducible** on any machine.

In [ ]:
# Find the model folder
model_path = f'mlruns/0/{run_id}/artifacts/iris-model'

# Show conda.yaml
conda_yaml_path = os.path.join(model_path, 'conda.yaml')
if os.path.exists(conda_yaml_path):
    print('📄 conda.yaml (auto-generated by MLflow):')  
    print('-' * 50)
    with open(conda_yaml_path) as f:
        print(f.read())
else:
    print(f'Path not found: {conda_yaml_path}')
    # Try alternate path structure
    import glob
    paths = glob.glob('mlruns/**/**/iris-model/conda.yaml', recursive=True)
    if paths:
        with open(paths[0]) as f:
            print(f.read())

In [ ]:
# Show requirements.txt
import glob
req_paths = glob.glob('mlruns/**/**/iris-model/requirements.txt', recursive=True)
if req_paths:
    print('📄 requirements.txt (auto-generated by MLflow):')
    print('-' * 50)
    with open(req_paths[0]) as f:
        print(f.read())

## Step 6: Load Saved Model & Predict

In [ ]:
# Load model from MLflow registry
loaded_model = mlflow.sklearn.load_model(f'runs:/{run_id}/iris-model')

# Test predictions
samples = {
    'Setosa':     [5.1, 3.5, 1.4, 0.2],
    'Versicolor': [6.4, 3.2, 4.5, 1.5],
    'Virginica':  [6.3, 3.3, 6.0, 2.5]
}
class_names = list(iris.target_names)

print('\n🔮 Predictions from Loaded Model:')
print('-' * 55)
for true_label, features in samples.items():
    pred_idx  = loaded_model.predict([features])[0]
    pred_name = class_names[pred_idx]
    proba     = loaded_model.predict_proba([features])[0]
    confidence = proba[pred_idx]
    status = '✅' if pred_name.lower() == true_label.lower() else '❌'
    print(f'  {status} Expected: {true_label:12s} → Predicted: {pred_name:12s} (conf: {confidence:.2%})')

## Step 7: Test the API (run app.py first!)

In a **separate terminal**, run:
```
python app.py
```
Then execute the cell below.

In [ ]:
import requests

try:
    # Health check
    resp = requests.get('http://localhost:8000/', timeout=3)
    print('Health check:', resp.json())

    # Prediction
    payload = {'sepal_length': 5.1, 'sepal_width': 3.5, 'petal_length': 1.4, 'petal_width': 0.2}
    resp = requests.post('http://localhost:8000/predict', json=payload, timeout=3)
    print('\nPrediction result:')
    print(json.dumps(resp.json(), indent=2))
except requests.exceptions.ConnectionError:
    print('⚠️  Server not running. Start app.py in a separate terminal first.')

## Step 8: Docker Commands (for reference)

Run these in your terminal after completing Steps 1-4:

```bash
# Build Docker image
docker build -t iris-mlflow-api .

# Run Docker container
docker run -p 8000:8000 iris-mlflow-api

# Verify reproducibility
python test_api.py
```